In [ ]:
!pip -q install -U albumentations pycocotools ultralytics

import os
import io
import time
import math
import json
import glob
import random
import contextlib
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import re
from IPython.display import display

import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision.models.detection.fcos import FCOS_ResNet50_FPN_Weights, FCOSClassificationHead
import albumentations as A

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import RTDETR
from google.colab import drive

CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
NUM_CLASSES = len(CLASSES) + 1
IMG_SIZE = 1024
GRAY = 114
FRAME_STEP = 3
SEED = 42

CKPT_DIR = '/content/drive/MyDrive/visdrone_ckpt'

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Device: {device}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Device: cuda


In [ ]:
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/multiple-object-video-detection-hw-2-ucu-ai-2026.zip'
EXTRACT_DIR = '/content/dataset'

if not os.path.exists(EXTRACT_DIR):
    print("Розпакування архіву у середовище Colab...")
    !unzip -q "{ZIP_PATH}" -d "{EXTRACT_DIR}"
    print("Розпакування успішно завершено!")

ROOT = EXTRACT_DIR
CKPT_DIR = '/content/drive/MyDrive/visdrone_ckpt'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Шлях до даних (ROOT): {ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Розпакування архіву у середовище Colab...
Розпакування успішно завершено!
Шлях до даних (ROOT): /content/dataset


Читаємо COCO-анотації в таблиці. `ioa_boxes` рахує, яка частка об'єкта лежить в ignored-зоні (`iscrowd=1`).
`VisDroneDet` заливає ignored-зони сірим, відкидає об'єкти, що більш ніж на 50% у зоні, і зменшує кадр до 1024 по довшій стороні.
Аугментації (тільки для train): горизонтальне віддзеркалення, масштаб 0.8–1.25, зсув ±10%, зміна яскравості/контрасту/насиченості.

In [ ]:
# Створення DataFrames з COCO JSON
def load_coco_as_df(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    imgs = pd.DataFrame(data['images'])
    anns = pd.DataFrame(data['annotations'])

    imgs['video'] = imgs['file_name'].apply(lambda x: x.rsplit('/', 1)[0] if '/' in x else x.rsplit('_', 1)[0])
    imgs['frame_id'] = imgs['file_name'].apply(lambda x: x.rsplit('/', 1)[-1] if '/' in x else x)
    return imgs, anns

# Перекриття боксів з ігнорованими зонами
def ioa_boxes(objs, zones):
    o = np.asarray(objs, float); z = np.asarray(zones, float)
    iw = np.clip(np.minimum((o[:,0]+o[:,2])[:,None], (z[:,0]+z[:,2])[None]) -
                 np.maximum(o[:,0][:,None], z[:,0][None]), 0, None)
    ih = np.clip(np.minimum((o[:,1]+o[:,3])[:,None], (z[:,1]+z[:,3])[None]) -
                 np.maximum(o[:,1][:,None], z[:,1][None]), 0, None)
    return ((iw*ih) / (o[:,2]*o[:,3])[:,None]).max(axis=1)

# Аугментації Albumentations
TRAIN_AUG = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.8, 1.25), translate_percent=(-0.1, 0.1),
                 fit_output=False, border_mode=cv2.BORDER_CONSTANT,
                 fill=GRAY, p=0.5),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.3, hue=0.0, p=0.8),
    ],
    bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'],
                             min_visibility=0.4, min_area=16),
)

# Клас Dataset
class VisDroneDet(Dataset):
    def __init__(self, split, imgs, anns, frame_step=1, train=False):
        self.split, self.train = split, train
        imgs = imgs.sort_values(['video', 'frame_id'])
        imgs = imgs.groupby('video', group_keys=False).apply(lambda d: d.iloc[::frame_step])
        self.imgs = imgs.reset_index(drop=True)
        self.by_img = {k: g for k, g in anns.groupby('image_id')}

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        row = self.imgs.iloc[i]
        img = cv2.cvtColor(cv2.imread(f"{ROOT}/{self.split}/{row['file_name']}"), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        g = self.by_img.get(row['id'])
        boxes = np.zeros((0, 4), np.float32); labels = np.zeros((0,), np.int64)
        if g is not None:
            zones = g[g['iscrowd'] == 1]['bbox'].tolist()
            objs = g[g['iscrowd'] == 0]
            ob = objs['bbox'].tolist()
            lb = objs['category_id'].values
            if zones:
                for x, y, w, h in zones:
                    img[int(y):int(y+h), int(x):int(x+w)] = GRAY
                if ob:
                    keep = ioa_boxes(ob, zones) <= 0.5
                    ob = [b for b, k in zip(ob, keep) if k]; lb = lb[keep]
            if ob:
                b = np.asarray(ob, np.float32)
                boxes = np.stack([b[:,0], b[:,1], b[:,0]+b[:,2], b[:,1]+b[:,3]], 1)
                labels = lb.astype(np.int64)

        s = IMG_SIZE / max(H, W)
        img = cv2.resize(img, (round(W*s), round(H*s)), interpolation=cv2.INTER_AREA)
        boxes = boxes * s
        nh, nw = img.shape[:2]

        if self.train:
            boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, nw)
            boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, nh)
            ok = ((boxes[:, 2] - boxes[:, 0]) > 1) & ((boxes[:, 3] - boxes[:, 1]) > 1)
            boxes, labels = boxes[ok], labels[ok]
            out = TRAIN_AUG(image=img, bboxes=boxes.tolist(), labels=labels.tolist())
            img = out['image']
            boxes = np.asarray(out['bboxes'], np.float32).reshape(-1, 4)
            labels = np.asarray(out['labels'], np.int64)

        t = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 255.0
        target = {
            'boxes': torch.from_numpy(boxes).float().reshape(-1, 4),
            'labels': torch.from_numpy(labels),
            'image_id': int(row['id']),
            'scale': s
        }
        return t, target

def collate(b):
    return tuple(zip(*b))

Train: кожен 3-й кадр кожного відео (сусідні кадри майже однакові), всього 8090 кадрів. Val: усі 2846 кадрів.
Фіксуємо seed для воркерів і перемішування, щоб навчання було відтворюваним.

In [ ]:
train_imgs, train_anns = load_coco_as_df(f'{ROOT}/train/annotations_coco.json')
val_imgs, val_anns = load_coco_as_df(f'{ROOT}/val/annotations_coco.json')

train_ds = VisDroneDet('train', train_imgs, train_anns, frame_step=FRAME_STEP, train=True)
val_ds   = VisDroneDet('val',   val_imgs,   val_anns,   frame_step=1, train=False)
print('train кадрів:', len(train_ds), '| val кадрів:', len(val_ds))

def seed_worker(worker_id):
    s = SEED + worker_id
    np.random.seed(s); random.seed(s)

g = torch.Generator(); g.manual_seed(SEED)

BATCH = 8
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2,
                      collate_fn=collate, worker_init_fn=seed_worker, generator=g)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, collate_fn=collate)

/tmp/ipykernel_1565/3185876980.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  imgs = imgs.groupby('video', group_keys=False).apply(lambda d: d.iloc[::frame_step])


train кадрів: 8090 | val кадрів: 2846


/tmp/ipykernel_1565/3185876980.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  imgs = imgs.groupby('video', group_keys=False).apply(lambda d: d.iloc[::frame_step])


Беремо FCOS ResNet50-FPN, заздалегідь навчений на COCO, і замінюємо класифікаційну голову під наші 8 класів + фон.

In [ ]:
def build_fcos_model():
    m = torchvision.models.detection.fcos_resnet50_fpn(
        weights=FCOS_ResNet50_FPN_Weights.DEFAULT,
        min_size=576, max_size=1024,
        box_detections_per_img=300
    )

    in_channels = m.head.classification_head.conv[0].in_channels
    num_anchors = m.head.classification_head.num_anchors

    m.head.classification_head = FCOSClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=NUM_CLASSES
    )
    return m

model = build_fcos_model()
print("FCOS модель успішно згенерована.")

Downloading: "https://download.pytorch.org/models/fcos_resnet50_fpn_coco-99b0c9b7.pth" to /root/.cache/torch/hub/checkpoints/fcos_resnet50_fpn_coco-99b0c9b7.pth


100%|██████████| 124M/124M [00:00<00:00, 162MB/s]


FCOS модель успішно згенерована.


`evaluate` запускає модель на val, повертає рамки в оригінальні координати і через COCOeval рахує
mAP (IoU 0.50:0.95), AP50, AP_small (об'єкти < 32×32 px) та FPS.

In [ ]:
val_gt = COCO(f'{ROOT}/val/annotations_coco.json')

def ap_from_eval(E, iou=None, area='all', cls=None):
    p = E.eval['precision']
    a = {'all': 0, 'small': 1, 'medium': 2, 'large': 3}[area]
    p = p[:, :, :, a, -1]
    if cls is not None:
        p = p[:, :, cls:cls+1]
    if iou is not None:
        t = int(np.where(np.isclose(E.params.iouThrs, iou))[0][0])
        p = p[t:t+1]
    p = p[p > -1]
    return float(p.mean()) if p.size else float('nan')

@torch.no_grad()
def evaluate(model, dl, ds, max_batches=None):
    model.eval()
    results, img_ids = [], []
    t_inf, n_img = 0.0, 0
    for bi, (imgs, tgts) in enumerate(dl):
        if max_batches and bi >= max_batches:
            break
        imgs_d = [i.to(device) for i in imgs]
        torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast('cuda'):
            outs = model(imgs_d)
        torch.cuda.synchronize(); t_inf += time.time() - t0
        n_img += len(imgs)
        for o, t in zip(outs, tgts):
            s = t['scale']; iid = t['image_id']; img_ids.append(iid)
            b = o['boxes'].float().cpu().numpy() / s
            for (x1, y1, x2, y2), sc, lb in zip(b, o['scores'].float().cpu().numpy(), o['labels'].cpu().numpy()):
                results.append({'image_id': iid, 'category_id': int(lb),
                                'bbox': [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                                'score': float(sc)})
    fps = n_img / max(t_inf, 1e-9)
    if not results:
        return {'mAP': 0, 'AP50': 0, 'AP_small': 0, 'fps': fps}
    dt = val_gt.loadRes(results)
    E = COCOeval(val_gt, dt, 'bbox')
    E.params.imgIds = img_ids
    E.params.maxDets = [1, 10, 300]
    with contextlib.redirect_stdout(io.StringIO()):
        E.evaluate(); E.accumulate()
    return {'mAP': ap_from_eval(E),
            'AP50': ap_from_eval(E, iou=0.5),
            'AP_small': ap_from_eval(E, area='small'),
            'fps': fps, 'eval': E}

loading annotations into memory...
Done (t=0.48s)
creating index...
index created!


SGD (lr 0.01, momentum 0.9), warmup на 300 ітерацій, далі косинусне спадання LR, mixed precision, обрізання градієнтів.
Після кожної епохи ваги зберігаються на Drive і рахуються метрики на val.

In [ ]:
EPOCHS = 3
LR = 0.01
WARMUP_ITERS = 300

def train_model(model, name, epochs=EPOCHS, lr=LR):
    model.to(device).train()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda')

    total_iters = epochs * len(train_dl)

    def lr_at(it):
        if it < WARMUP_ITERS:
            return lr * (0.001 + 0.999 * it / WARMUP_ITERS)
        p = (it - WARMUP_ITERS) / max(1, total_iters - WARMUP_ITERS)
        return lr * 0.5 * (1 + math.cos(math.pi * p))

    it = 0
    history = []
    for ep in range(epochs):
        model.train()
        t0 = time.time()
        run = {}
        for imgs, tgts in train_dl:
            for gr in opt.param_groups:
                gr['lr'] = lr_at(it)
            imgs = [i.to(device) for i in imgs]
            tgts = [{'boxes': t['boxes'].to(device), 'labels': t['labels'].to(device)} for t in tgts]

            with torch.autocast('cuda'):
                losses = model(imgs, tgts)
                loss = sum(losses.values())

            if not torch.isfinite(loss):
                print('нескінченний loss, пропускаю ітерацію', it)
                opt.zero_grad(); it += 1
                continue

            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 10.0)
            scaler.step(opt); scaler.update()

            for k, v in losses.items():
                run[k] = run.get(k, 0) + v.item()
            it += 1
            if it % 50 == 0:
                n = 50
                print(f'ep {ep+1} it {it}/{total_iters} lr {lr_at(it):.5f} | ' +
                      ' '.join(f'{k[5:] if k.startswith("loss_") else k}={run[k]/n:.3f}' for k in run) +
                      f' | {time.time()-t0:.0f}s')
                history.append({'it': it, **{k: run[k]/n for k in run}})
                run = {}

        path = f'{CKPT_DIR}/{name}_ep{ep+1}.pt'
        torch.save({'model': model.state_dict(), 'epoch': ep + 1}, path)
        print(f'епоха {ep+1} завершена за {(time.time()-t0)/60:.1f} хв, збережено {path}')
        r = evaluate(model, val_dl, val_ds)
        print(f"  VAL ep{ep+1}: mAP={r['mAP']:.3f} AP50={r['AP50']:.3f} "
              f"AP_small={r['AP_small']:.3f} fps={r['fps']:.1f}")
        model.train()
    return history

Та сама модель, але тепер можна змінювати параметри постобробки: поріг NMS,
кількість кандидатів до NMS (`topk`) і максимум детекцій на кадр.
`run_experiment` навчає модель із заданими параметрами й повертає метрики на val.

In [ ]:
def build_fcos_model(
    num_classes=NUM_CLASSES,
    min_size=576,
    max_size=1024,
    box_score_thresh=0.05,
    box_nms_thresh=0.6,
    box_topk_candidates=1000,
    box_detections_per_img=300
):
    m = torchvision.models.detection.fcos_resnet50_fpn(
        weights=FCOS_ResNet50_FPN_Weights.DEFAULT,
        min_size=min_size,
        max_size=max_size,
        box_score_thresh=box_score_thresh,
        box_nms_thresh=box_nms_thresh,
        box_topk_candidates=box_topk_candidates,
        box_detections_per_img=box_detections_per_img
    )

    in_channels = m.head.classification_head.conv[0].in_channels
    num_anchors = m.head.classification_head.num_anchors

    m.head.classification_head = FCOSClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes
    )
    return m

def run_experiment(exp_name, model_params, epochs=EPOCHS, lr=LR):
    print(f"\n" + "="*40)
    print(f" ПОЧАТОК ЕКСПЕРИМЕНТУ: {exp_name}")
    print(f" Параметри: {model_params}")
    print("="*40 + "\n")

    exp_model = build_fcos_model(**model_params)

    history = train_model(exp_model, name=exp_name, epochs=epochs, lr=lr)

    val_res = evaluate(exp_model, val_dl, val_ds)

    result_summary = {
        'Experiment': exp_name,
        'mAP': val_res['mAP'],
        'AP50': val_res['AP50'],
        'AP_small': val_res['AP_small'],
        'FPS': val_res['fps'],
        **model_params
    }

    return exp_model, result_summary

In [ ]:
experiments_results = []

# Експеримент 1: Бейзлайн (стандартні параметри: topk=1000, detections=300)
params_exp1 = {
    'box_topk_candidates': 1000,
    'box_detections_per_img': 300,
    'box_nms_thresh': 0.6
}
_, res1 = run_experiment('fcos_baseline', params_exp1, epochs=EPOCHS, lr=LR)
experiments_results.append(res1)

# Експеримент 2: Збільшена кількість кандидатів для щільних сцен VisDrone (topk=2000, detections=500)
params_exp2 = {
    'box_topk_candidates': 2000,
    'box_detections_per_img': 500,
    'box_nms_thresh': 0.6
}
_, res2 = run_experiment('fcos_dense_candidates', params_exp2, epochs=EPOCHS, lr=LR)
experiments_results.append(res2)

# Експеримент 3: Тюнінг порогу NMS (nms_thresh=0.5 для зменшення дублюючих рамок)
params_exp3 = {
    'box_topk_candidates': 2000,
    'box_detections_per_img': 500,
    'box_nms_thresh': 0.5
}
_, res3 = run_experiment('fcos_nms_0_5', params_exp3, epochs=EPOCHS, lr=LR)
experiments_results.append(res3)

df_results = pd.DataFrame(experiments_results)
print("\n--- ПОРІВНЯЛЬНА ТАБЛИЦЯ ЕКСПЕРИМЕНТІВ ---")
display(df_results[['Experiment', 'mAP', 'AP50', 'AP_small', 'FPS', 'box_topk_candidates', 'box_detections_per_img', 'box_nms_thresh']])


 ПОЧАТОК ЕКСПЕРИМЕНТУ: fcos_baseline
 Параметри: {'box_topk_candidates': 1000, 'box_detections_per_img': 300, 'box_nms_thresh': 0.6}

ep 1 it 50/3036 lr 0.00168 | classification=0.647 bbox_regression=0.358 bbox_ctrness=0.639 | 42s
ep 1 it 100/3036 lr 0.00334 | classification=0.333 bbox_regression=0.342 bbox_ctrness=0.635 | 82s
ep 1 it 150/3036 lr 0.00500 | classification=0.274 bbox_regression=0.327 bbox_ctrness=0.633 | 120s
ep 1 it 200/3036 lr 0.00667 | classification=0.257 bbox_regression=0.320 bbox_ctrness=0.631 | 160s
ep 1 it 250/3036 lr 0.00834 | classification=0.246 bbox_regression=0.315 bbox_ctrness=0.631 | 198s
ep 1 it 300/3036 lr 0.01000 | classification=0.240 bbox_regression=0.318 bbox_ctrness=0.629 | 237s
ep 1 it 350/3036 lr 0.00999 | classification=0.223 bbox_regression=0.313 bbox_ctrness=0.629 | 276s
ep 1 it 400/3036 lr 0.00997 | classification=0.220 bbox_regression=0.309 bbox_ctrness=0.628 | 315s
ep 1 it 450/3036 lr 0.00993 | classification=0.204 bbox_regression=0.302 bbo

Вибираємо 1500 випадкових train-кадрів (фіксований seed: ті самі для всіх моделей) без аугментацій.
Оцінка на train і val показує, чи модель перенавчається.

In [ ]:
EVAL_EXPERIMENTS = {
    'fcos_baseline':         {'box_topk_candidates': 1000, 'box_detections_per_img': 300, 'box_nms_thresh': 0.6},
    'fcos_dense_candidates': {'box_topk_candidates': 2000, 'box_detections_per_img': 500, 'box_nms_thresh': 0.6},
    'fcos_nms_0_5':          {'box_topk_candidates': 2000, 'box_detections_per_img': 500, 'box_nms_thresh': 0.5},
}
EVAL_EPOCHS = [1, 2, 3]
N_TRAIN_EVAL = 1500

OUT_CSV_LONG = f'{CKPT_DIR}/fcos_per_epoch_metrics.csv'
OUT_CSV_WIDE = f'{CKPT_DIR}/fcos_per_epoch_metrics_wide.csv'

print('Чекпойнти FCOS у', CKPT_DIR)
for p in sorted(glob.glob(f'{CKPT_DIR}/fcos_*.pt')):
    print('  ', os.path.basename(p))


train_gt = COCO(f'{ROOT}/train/annotations_coco.json')

train_eval_full = VisDroneDet('train', train_imgs, train_anns, frame_step=FRAME_STEP, train=False)
rng = np.random.RandomState(SEED)
train_eval_idx = np.sort(rng.choice(len(train_eval_full), size=N_TRAIN_EVAL, replace=False))
train_eval_ds = Subset(train_eval_full, train_eval_idx.tolist())
train_eval_dl = DataLoader(train_eval_ds, batch_size=BATCH, shuffle=False,
                           num_workers=2, collate_fn=collate)

Чекпойнти FCOS у /content/drive/MyDrive/visdrone_ckpt
   fcos_ciou_ep1.pt
   fcos_ciou_ep2.pt
   fcos_ciou_ep3.pt
   fcos_radius25_ep1.pt
   fcos_radius25_ep2.pt
   fcos_radius25_ep3.pt
loading annotations into memory...
Done (t=7.08s)
creating index...
index created!


/tmp/ipykernel_1565/3185876980.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  imgs = imgs.groupby('video', group_keys=False).apply(lambda d: d.iloc[::frame_step])


In [ ]:
print(f'Train-підмножина: {len(train_eval_ds)} кадрів з {len(train_eval_full)} | Val: {len(val_ds)} кадрів')


@torch.no_grad()
def evaluate_split(model, dl, gt_coco):
    model.eval()
    results, img_ids = [], []
    for imgs, tgts in dl:
        imgs_d = [i.to(device) for i in imgs]
        with torch.autocast('cuda'):
            outs = model(imgs_d)
        for o, t in zip(outs, tgts):
            s, iid = t['scale'], t['image_id']
            img_ids.append(iid)
            b = o['boxes'].float().cpu().numpy() / s
            for (x1, y1, x2, y2), sc, lb in zip(b, o['scores'].float().cpu().numpy(),
                                                o['labels'].cpu().numpy()):
                results.append({'image_id': iid, 'category_id': int(lb),
                                'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                                'score': float(sc)})
    if not results:
        return {'mAP': 0.0, 'AP50': 0.0, 'AP_small': 0.0}
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt_coco.loadRes(results)
        E = COCOeval(gt_coco, dt, 'bbox')
        E.params.imgIds = img_ids
        E.params.maxDets = [1, 10, 300]
        E.evaluate(); E.accumulate()
    return {'mAP':      ap_from_eval(E),
            'AP50':     ap_from_eval(E, iou=0.5),
            'AP_small': ap_from_eval(E, area='small')}


if os.path.exists(OUT_CSV_LONG):
    df_long = pd.read_csv(OUT_CSV_LONG)
    print(f'Знайдено попередні результати: {len(df_long)} рядків, продовжую')
else:
    df_long = pd.DataFrame(columns=['Experiment', 'Epoch', 'Split', 'mAP', 'AP50', 'AP_small'])

def already_done(exp, ep, split):
    return ((df_long['Experiment'] == exp) & (df_long['Epoch'] == ep) &
            (df_long['Split'] == split)).any()

SPLITS = [('Train (1500 rand)', train_eval_dl, train_gt),
          ('Val (full)',        val_dl,        val_gt)]

for exp_name, params in EVAL_EXPERIMENTS.items():
    for ep in EVAL_EPOCHS:
        ckpt_path = f'{CKPT_DIR}/{exp_name}_ep{ep}.pt'
        if not os.path.exists(ckpt_path):
            print(f'[ПРОПУСК] немає файлу {ckpt_path}')
            continue
        todo = [s for s in SPLITS if not already_done(exp_name, ep, s[0])]
        if not todo:
            continue

        fcos_m = build_fcos_model(**params)
        fcos_m.load_state_dict(torch.load(ckpt_path, map_location=device)['model'])
        fcos_m.to(device)

        for split_name, dl, gt in todo:
            t0 = time.time()
            r = evaluate_split(fcos_m, dl, gt)
            row = {'Experiment': exp_name, 'Epoch': ep, 'Split': split_name,
                   'mAP': round(r['mAP'], 4), 'AP50': round(r['AP50'], 4),
                   'AP_small': round(r['AP_small'], 4)}
            df_long = pd.concat([df_long, pd.DataFrame([row])], ignore_index=True)
            df_long.to_csv(OUT_CSV_LONG, index=False)
            print(f"{exp_name:22s} ep{ep} | {split_name:18s} | mAP={row['mAP']:.4f} "
                  f"AP50={row['AP50']:.4f} AP_small={row['AP_small']:.4f} | {time.time()-t0:.0f}s")

        del fcos_m
        torch.cuda.empty_cache()


df_long['Epoch'] = df_long['Epoch'].astype(int)
df_long = df_long.sort_values(['Experiment', 'Epoch', 'Split']).reset_index(drop=True)

df_wide = df_long.pivot_table(index=['Experiment', 'Epoch'], columns='Split',
                              values=['mAP', 'AP50', 'AP_small'])
df_wide.columns = [f'{split} {metric}' for metric, split in df_wide.columns]
order = [f'{s} {m}' for s in ['Train (1500 rand)', 'Val (full)'] for m in ['mAP', 'AP50', 'AP_small']]
df_wide = df_wide[[c for c in order if c in df_wide.columns]].reset_index()

df_long.to_csv(OUT_CSV_LONG, index=False)
df_wide.to_csv(OUT_CSV_WIDE, index=False)

print('\n' + '=' * 100)
print('FCOS: mAP / AP50 / AP_small по епохах (Train = 1500 випадкових кадрів, Val = повний)')
print('=' * 100)
print(df_wide.to_string(index=False))
print('=' * 100)
print(f'Збережено на Drive:\n  {OUT_CSV_LONG}\n  {OUT_CSV_WIDE}')
display(df_wide)

Чекпойнти FCOS у /content/drive/MyDrive/visdrone_ckpt
   fcos_baseline_ep1.pt
   fcos_baseline_ep2.pt
   fcos_baseline_ep3.pt
   fcos_dense_candidates_ep1.pt
   fcos_dense_candidates_ep2.pt
   fcos_dense_candidates_ep3.pt
   fcos_nms_0_5_ep1.pt
   fcos_nms_0_5_ep2.pt
   fcos_nms_0_5_ep3.pt
   fcos_resnet50_fpn_ep1.pt
   fcos_resnet50_fpn_ep2.pt
   fcos_resnet50_fpn_ep3.pt
loading annotations into memory...
Done (t=4.98s)
creating index...
index created!


/tmp/ipykernel_712/3220585706.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  imgs = imgs.groupby('video', group_keys=False).apply(lambda d: d.iloc[::frame_step])


Train-підмножина: 1500 кадрів з 8090 | Val: 2846 кадрів
fcos_baseline          ep1 | Train (1500 rand)  | mAP=0.3209 AP50=0.5710 AP_small=0.1394 | 141s


/tmp/ipykernel_712/322620437.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_long = pd.concat([df_long, pd.DataFrame([row])], ignore_index=True)


fcos_baseline          ep1 | Val (full)         | mAP=0.1477 AP50=0.2891 AP_small=0.0653 | 320s
fcos_baseline          ep2 | Train (1500 rand)  | mAP=0.4010 AP50=0.6928 AP_small=0.2218 | 150s
fcos_baseline          ep2 | Val (full)         | mAP=0.1451 AP50=0.2877 AP_small=0.0606 | 315s
fcos_baseline          ep3 | Train (1500 rand)  | mAP=0.4193 AP50=0.7125 AP_small=0.2392 | 151s
fcos_baseline          ep3 | Val (full)         | mAP=0.1466 AP50=0.2886 AP_small=0.0636 | 318s
fcos_dense_candidates  ep1 | Train (1500 rand)  | mAP=0.3144 AP50=0.5664 AP_small=0.1655 | 150s
fcos_dense_candidates  ep1 | Val (full)         | mAP=0.1412 AP50=0.2779 AP_small=0.0666 | 320s
fcos_dense_candidates  ep2 | Train (1500 rand)  | mAP=0.3993 AP50=0.6885 AP_small=0.2268 | 148s
fcos_dense_candidates  ep2 | Val (full)         | mAP=0.1516 AP50=0.2969 AP_small=0.0636 | 320s
fcos_dense_candidates  ep3 | Train (1500 rand)  | mAP=0.4175 AP50=0.7125 AP_small=0.2383 | 148s
fcos_dense_candidates  ep3 | Val (full) 

,Experiment,Epoch,Train (1500 rand) mAP,Train (1500 rand) AP50,Train (1500 rand) AP_small,Val (full) mAP,Val (full) AP50,Val (full) AP_small
0,fcos_baseline,1,0.3209,0.5710,0.1394,0.1477,0.2891,0.0653
1,fcos_baseline,2,0.4010,0.6928,0.2218,0.1451,0.2877,0.0606
2,fcos_baseline,3,0.4193,0.7125,0.2392,0.1466,0.2886,0.0636
3,fcos_dense_candidates,1,0.3144,0.5664,0.1655,0.1412,0.2779,0.0666
4,fcos_dense_candidates,2,0.3993,0.6885,0.2268,0.1516,0.2969,0.0636
5,fcos_dense_candidates,3,0.4175,0.7125,0.2383,0.1518,0.2977,0.0640
6,fcos_nms_0_5,1,0.3218,0.5797,0.1351,0.1399,0.2805,0.0539
7,fcos_nms_0_5,2,0.4054,0.6944,0.2294,0.1521,0.2999,0.0618
8,fcos_nms_0_5,3,0.4196,0.7156,0.2358,0.1477,0.2910,0.0653


### Експерименти з навчанням: box regression loss і label assignment

- **`fcos_ciou`**: замінюємо функцію втрат для рамок GIoU → **CIoU**. CIoU додатково штрафує за відстань між центрами
  і різницю пропорцій рамок, що має допомогти точніше локалізувати дрібні об'єкти.
- **`fcos_radius25`**: збільшуємо радіус center sampling 1.5 → **2.5**. Більше точок навколо центру об'єкта
  вважаються «позитивними», тож малі об'єкти отримують більше навчальних прикладів.

In [ ]:
import torchvision.models.detection.fcos as fcos_mod
from torchvision.ops import complete_box_iou_loss

FCOS_POST = dict(score_thresh=0.05, nms_thresh=0.6, topk_candidates=1000, detections_per_img=300)

def build_fcos(center_sampling_radius=1.5, **post):
    m = build_fcos_model()
    m.center_sampling_radius = center_sampling_radius
    for k, v in {**FCOS_POST, **post}.items():
        setattr(m, k, v)
    return m

In [ ]:
def fresh_loader():
    global train_dl
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    gen = torch.Generator(); gen.manual_seed(SEED)
    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2,
                          collate_fn=collate, worker_init_fn=seed_worker, generator=gen)

def ciou_loss(pred, target, reduction='sum'):
    return complete_box_iou_loss(pred.float(), target.float(), reduction=reduction)

FCOS_TRAIN_EXPS = {
    'fcos_ciou':     {'loss': 'ciou', 'radius': 1.5},
    'fcos_radius25': {'loss': 'giou', 'radius': 2.5},
}

for name, cfg in FCOS_TRAIN_EXPS.items():
    if os.path.exists(f'{CKPT_DIR}/{name}_ep{EPOCHS}.pt'):
        print(f'{name}: вже навчено, пропускаю')
        continue
    print('\n' + '=' * 40 + f'\n {name}: {cfg}\n' + '=' * 40)
    fresh_loader()
    exp_model = build_fcos(center_sampling_radius=cfg['radius'])
    orig_loss = fcos_mod.generalized_box_iou_loss
    if cfg['loss'] == 'ciou':
        fcos_mod.generalized_box_iou_loss = ciou_loss
    try:
        train_model(exp_model, name=name)
    finally:
        fcos_mod.generalized_box_iou_loss = orig_loss
    del exp_model; torch.cuda.empty_cache()


 fcos_ciou: {'loss': 'ciou', 'radius': 1.5}
ep 1 it 50/3036 lr 0.00168 | classification=0.624 bbox_regression=0.353 bbox_ctrness=0.640 | 42s
ep 1 it 100/3036 lr 0.00334 | classification=0.311 bbox_regression=0.337 bbox_ctrness=0.635 | 78s
ep 1 it 150/3036 lr 0.00500 | classification=0.270 bbox_regression=0.322 bbox_ctrness=0.633 | 115s
ep 1 it 200/3036 lr 0.00667 | classification=0.272 bbox_regression=0.314 bbox_ctrness=0.630 | 151s
ep 1 it 250/3036 lr 0.00834 | classification=0.248 bbox_regression=0.311 bbox_ctrness=0.631 | 188s
ep 1 it 300/3036 lr 0.01000 | classification=0.247 bbox_regression=0.315 bbox_ctrness=0.630 | 225s
ep 1 it 350/3036 lr 0.00999 | classification=0.226 bbox_regression=0.309 bbox_ctrness=0.629 | 262s
ep 1 it 400/3036 lr 0.00997 | classification=0.214 bbox_regression=0.302 bbox_ctrness=0.628 | 299s
ep 1 it 450/3036 lr 0.00993 | classification=0.202 bbox_regression=0.295 bbox_ctrness=0.628 | 336s
ep 1 it 500/3036 lr 0.00987 | classification=0.211 bbox_regression=

In [ ]:
@torch.no_grad()
def eval_sizes(model, dl, gt):
    model.eval()
    results, img_ids = [], []
    t_inf, n_img = 0.0, 0
    for imgs, tgts in dl:
        imgs_d = [i.to(device) for i in imgs]
        torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast('cuda'):
            outs = model(imgs_d)
        torch.cuda.synchronize(); t_inf += time.time() - t0
        n_img += len(imgs)
        for o, t in zip(outs, tgts):
            s, iid = t['scale'], t['image_id']
            img_ids.append(iid)
            b = o['boxes'].float().cpu().numpy() / s
            for (x1, y1, x2, y2), sc, lb in zip(b, o['scores'].float().cpu().numpy(), o['labels'].cpu().numpy()):
                results.append({'image_id': iid, 'category_id': int(lb),
                                'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                                'score': float(sc)})
    out = {'FPS': n_img / max(t_inf, 1e-9)}
    if not results:
        return {**out, 'mAP': 0.0, 'AP50': 0.0, 'AP_small': 0.0, 'AP_medium': 0.0, 'AP_large': 0.0}
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(results)
        E = COCOeval(gt, dt, 'bbox')
        E.params.imgIds = img_ids
        E.params.maxDets = [1, 10, 300]
        E.evaluate(); E.accumulate()
    out.update({'mAP': ap_from_eval(E), 'AP50': ap_from_eval(E, iou=0.5),
                'AP_small': ap_from_eval(E, area='small'),
                'AP_medium': ap_from_eval(E, area='medium'),
                'AP_large': ap_from_eval(E, area='large')})
    return out

In [ ]:
FCOS_V2_EXPS = ['fcos_ciou', 'fcos_radius25']
OUT_CSV_V2 = f'{CKPT_DIR}/fcos_v2_per_epoch_metrics.csv'
df_v2 = pd.read_csv(OUT_CSV_V2) if os.path.exists(OUT_CSV_V2) else pd.DataFrame()

SPLITS_V2 = [('Train (1500 rand)', train_eval_dl, train_gt), ('Val (full)', val_dl, val_gt)]

for name in FCOS_V2_EXPS:
    for ep in [1, 2, 3]:
        ckpt_path = f'{CKPT_DIR}/{name}_ep{ep}.pt'
        if not os.path.exists(ckpt_path):
            print(f'[ПРОПУСК] немає {ckpt_path}'); continue
        todo = [s for s in SPLITS_V2 if not (len(df_v2) and ((df_v2['Experiment'] == name) &
                (df_v2['Epoch'] == ep) & (df_v2['Split'] == s[0])).any())]
        if not todo:
            continue
        m = build_fcos().to(device)
        m.load_state_dict(torch.load(ckpt_path, map_location=device)['model'])
        for split_name, dl, gt in todo:
            r = eval_sizes(m, dl, gt)
            row = {'Experiment': name, 'Epoch': ep, 'Split': split_name, **{k: round(v, 4) for k, v in r.items()}}
            df_v2 = pd.concat([df_v2, pd.DataFrame([row])], ignore_index=True)
            df_v2.to_csv(OUT_CSV_V2, index=False)
            print(f"{name:14s} ep{ep} | {split_name:18s} | mAP={r['mAP']:.4f} AP50={r['AP50']:.4f} AP_small={r['AP_small']:.4f}")
        del m; torch.cuda.empty_cache()

fcos_ciou      ep1 | Train (1500 rand)  | mAP=0.3343 AP50=0.6057 AP_small=0.1555
fcos_ciou      ep1 | Val (full)         | mAP=0.1502 AP50=0.3070 AP_small=0.0653
fcos_ciou      ep2 | Train (1500 rand)  | mAP=0.4094 AP50=0.7176 AP_small=0.2297
fcos_ciou      ep2 | Val (full)         | mAP=0.1601 AP50=0.3248 AP_small=0.0668
fcos_ciou      ep3 | Train (1500 rand)  | mAP=0.4286 AP50=0.7396 AP_small=0.2491
fcos_ciou      ep3 | Val (full)         | mAP=0.1585 AP50=0.3216 AP_small=0.0700
fcos_radius25  ep1 | Train (1500 rand)  | mAP=0.3180 AP50=0.5784 AP_small=0.1279
fcos_radius25  ep1 | Val (full)         | mAP=0.1413 AP50=0.2919 AP_small=0.0707
fcos_radius25  ep2 | Train (1500 rand)  | mAP=0.3941 AP50=0.6852 AP_small=0.1808
fcos_radius25  ep2 | Val (full)         | mAP=0.1498 AP50=0.3067 AP_small=0.0678
fcos_radius25  ep3 | Train (1500 rand)  | mAP=0.4108 AP50=0.7055 AP_small=0.2059
fcos_radius25  ep3 | Val (full)         | mAP=0.1505 AP50=0.3047 AP_small=0.0709


Перша таблиця: train і val по епохах та розрив між ними (`gap mAP`). Друга: val після 3-ї епохи за розміром об'єктів і FPS.
CIoU дав найкращий val mAP серед усіх FCOS (0.160 на 2-й епосі проти 0.152 у найкращого з попередніх),
а більший радіус трохи покращив AP_small (0.071). Розрив train/val росте з кожною епохою, тобто модель поступово перенавчається.

In [ ]:
wide = df_v2.pivot_table(index=['Experiment', 'Epoch'], columns='Split', values=['mAP', 'AP50', 'AP_small'])
wide.columns = [f'{s} {m}' for m, s in wide.columns]
wide['gap mAP'] = wide['Train (1500 rand) mAP'] - wide['Val (full) mAP']
display(wide.round(3))

size_tbl = (df_v2[(df_v2['Split'] == 'Val (full)') & (df_v2['Epoch'] == 3)]
            .set_index('Experiment')[['mAP', 'AP50', 'AP_small', 'AP_medium', 'AP_large', 'FPS']])
display(size_tbl.round(3))

Train (1500 rand) AP50  Val (full) AP50  \
Experiment    Epoch                                            
fcos_ciou     1                       0.606            0.307   
              2                       0.718            0.325   
              3                       0.740            0.322   
fcos_radius25 1                       0.578            0.292   
              2                       0.685            0.307   
              3                       0.706            0.305   

                     Train (1500 rand) AP_small  Val (full) AP_small  \
Experiment    Epoch                                                    
fcos_ciou     1                           0.156                0.065   
              2                           0.230                0.067   
              3                           0.249                0.070   
fcos_radius25 1                           0.128                0.071   
              2                           0.181                0.068   
              3                           0.206                0.071   

                     Train (1500 rand) mAP  Val (full) mAP  gap mAP  
Experiment    Epoch                                                  
fcos_ciou     1                      0.334           0.150    0.184  
              2                      0.409           0.160    0.249  
              3                      0.429           0.158    0.270  
fcos_radius25 1                      0.318           0.141    0.177  
              2                      0.394           0.150    0.244  
              3                      0.411           0.150    0.260

,mAP,AP50,AP_small,AP_medium,AP_large,FPS
Experiment,,,,,,
fcos_ciou,0.158,0.322,0.070,0.141,0.319,20.531
fcos_radius25,0.150,0.305,0.071,0.138,0.298,20.605
